# Advanced Problems with Solutions: Random Seeds and Reproducible Randomness in Python

This notebook contains advanced practice problems about reproducible pseudo-randomness using Python's `random` module.

## Learning goals

By the end, you should be able to:

- Explain why pseudo-random number generators are deterministic.
- Use `random.seed()` correctly for reproducible experiments.
- Avoid hidden bugs caused by shared global random state.
- Use independent random generators with `random.Random`.
- Build reproducible simulations.
- Test randomized code reliably.
- Analyze empirical frequencies using `collections.Counter`.
- Reason about how consuming random numbers changes future results.

## Recommended imports

In [1]:
import random
from collections import Counter
from statistics import mean, pstdev

## Problem 1: Reproducible Bug Hunt

A function below is supposed to simulate drawing three integers from 1 to 10.

However, a user reports that their tests sometimes fail because the result changes from run to run.

**Task**

1. Rewrite the function so it can be tested reproducibly.
2. Write a test showing that the same seed produces the same result.
3. Write a test showing that different seeds can produce different results.

### Starter code

In [2]:
def draw_three():
    return [random.randint(1, 10) for _ in range(3)]

### Solution

In [3]:
def draw_three(seed=None):
    rng = random.Random(seed)
    return [rng.randint(1, 10) for _ in range(3)]


# Same seed -> same result
result_a = draw_three(42)
result_b = draw_three(42)

assert result_a == result_b


# Different seeds usually produce different results.
# This exact comparison is safe here because these two known seeds produce different outputs.
assert draw_three(42) != draw_three(43)

print("Seed 42:", result_a)
print("Seed 43:", draw_three(43))

Seed 42: [2, 1, 5]
Seed 43: [1, 5, 3]


### Explanation

The key improvement is using a local `random.Random(seed)` instance instead of the global generator. This prevents the function from depending on unrelated random calls elsewhere in the program.

## Problem 2: Shared Global State Trap

The two functions below both use the global `random` module. This creates a subtle dependency: calling one function changes the output of the other.

**Task**

1. Demonstrate the problem.
2. Refactor the code so each function can be reproducible independently.

### Starter code

In [4]:
def random_name():
    names = ["Ada", "Grace", "Linus", "Guido"]
    return random.choice(names)


def random_score():
    return random.randint(0, 100)

### Solution

In [5]:
# Demonstration of the problem

random.seed(10)
score_without_name_first = random_score()

random.seed(10)
_ = random_name()
score_after_name_first = random_score()

print("Score without calling random_name first:", score_without_name_first)
print("Score after calling random_name first:", score_after_name_first)

assert score_without_name_first != score_after_name_first


# Refactor: inject a random generator into each function.

def random_name(rng):
    names = ["Ada", "Grace", "Linus", "Guido"]
    return rng.choice(names)


def random_score(rng):
    return rng.randint(0, 100)


name_rng = random.Random(10)
score_rng = random.Random(10)

name = random_name(name_rng)
score = random_score(score_rng)

print("Independent name:", name)
print("Independent score:", score)

# The score is now independent of whether the name generator was used.
score_rng_1 = random.Random(10)
score_rng_2 = random.Random(10)

_ = random_name(random.Random(999))
assert random_score(score_rng_1) == random_score(score_rng_2)

Score without calling random_name first: 73
Score after calling random_name first: 54
Independent name: Ada
Independent score: 73


### Explanation

Random generators have state. Every call consumes part of the sequence. Independent `Random` instances are the best practice when separate parts of a program should not affect each other.

## Problem 3: Predict the Sequence Consumption

The order of random calls matters.

Consider this function:

```python
def experiment(seed):
    random.seed(seed)
    a = random.randint(1, 5)
    b = random.random()
    c = random.randint(1, 5)
    return a, b, c
```

**Task**

Write another function that returns the same `c` value without storing `a` or `b`, but by consuming the same number of random values before generating `c`.

### Solution

In [6]:
def experiment(seed):
    random.seed(seed)
    a = random.randint(1, 5)
    b = random.random()
    c = random.randint(1, 5)
    return a, b, c


def reproduce_only_c(seed):
    random.seed(seed)
    random.randint(1, 5)  # consumes the same random state as a
    random.random()       # consumes the same random state as b
    return random.randint(1, 5)


for seed in range(5):
    a, b, c = experiment(seed)
    reproduced_c = reproduce_only_c(seed)
    print(seed, c, reproduced_c)
    assert c == reproduced_c

0 4 4
1 1 1
2 3 3
3 2 2
4 4 4


### Explanation

A pseudo-random generator produces a deterministic stream. Functions such as `randint`, `random`, `shuffle`, and `gauss` all advance the generator's internal state.

## Problem 4: Reproducible Shuffle Pipeline

You are building a reproducible data-splitting function.

**Task**

Write a function `train_test_split(items, test_size, seed)` that:

1. Does not mutate the original list.
2. Shuffles reproducibly.
3. Returns `(train, test)`.
4. Uses `test_size` as the number of test items.

### Solution

In [7]:
def train_test_split(items, test_size, seed=None):
    if test_size < 0 or test_size > len(items):
        raise ValueError("test_size must be between 0 and len(items)")
        
    rng = random.Random(seed)
    shuffled = list(items)  # copy to avoid mutating the original
    rng.shuffle(shuffled)
    
    test = shuffled[:test_size]
    train = shuffled[test_size:]
    return train, test


items = list(range(10))

train1, test1 = train_test_split(items, test_size=3, seed=123)
train2, test2 = train_test_split(items, test_size=3, seed=123)

assert items == list(range(10))  # original list was not changed
assert train1 == train2
assert test1 == test2
assert sorted(train1 + test1) == items
assert len(test1) == 3

print("Train:", train1)
print("Test:", test1)

Train: [9, 2, 3, 6, 1, 4, 0]
Test: [8, 7, 5]


### Explanation

A common mistake is calling `random.shuffle(items)` directly, which mutates the input list. Copying first is safer and easier to test.

## Problem 5: Frequency Analysis at Scale

Generate one million random integers between 0 and 10 inclusive.

**Task**

1. Use a fixed seed.
2. Count frequencies using `Counter`.
3. Compute the expected frequency for each integer.
4. Compute the largest absolute deviation from the expected frequency.
5. Explain whether the result seems reasonable.

### Solution

In [8]:
def frequency_report(low, high, n, seed=None):
    rng = random.Random(seed)
    values = [rng.randint(low, high) for _ in range(n)]
    counts = Counter(values)
    
    number_of_outcomes = high - low + 1
    expected = n / number_of_outcomes
    
    deviations = {
        value: counts[value] - expected
        for value in range(low, high + 1)
    }
    
    max_abs_deviation = max(abs(dev) for dev in deviations.values())
    
    return counts, expected, deviations, max_abs_deviation


counts, expected, deviations, max_abs_deviation = frequency_report(0, 10, 1_000_000, seed=0)

print("Counts:")
for value in sorted(counts):
    print(value, counts[value])

print("\nExpected per value:", expected)
print("Max absolute deviation:", max_abs_deviation)

Counts:
0 90935
1 91184
2 91002
3 91042
4 90766
5 91072
6 90678
7 90985
8 90409
9 91383
10 90544

Expected per value: 90909.09090909091
Max absolute deviation: 500.09090909091174


### Explanation

The distribution should be roughly even, but not perfectly even. Pseudo-random does not mean each outcome appears exactly the same number of times. With one million draws over eleven outcomes, deviations of hundreds are normal.

## Problem 6: Detecting a Bad Random Function

A teammate writes this function:

```python
def bad_die_roll():
    random.seed(100)
    return random.randint(1, 6)
```

They claim it is good because it is reproducible.

**Task**

1. Explain the bug.
2. Show the bug with code.
3. Write a corrected version.

### Solution

In [9]:
def bad_die_roll():
    random.seed(100)
    return random.randint(1, 6)


rolls = [bad_die_roll() for _ in range(10)]
print("Bad rolls:", rolls)

assert len(set(rolls)) == 1


def make_die_roller(seed=None):
    rng = random.Random(seed)
    
    def roll():
        return rng.randint(1, 6)
    
    return roll


roll = make_die_roller(seed=100)
good_rolls = [roll() for _ in range(10)]
print("Good reproducible rolls:", good_rolls)

assert len(set(good_rolls)) > 1
assert good_rolls == [make_die_roller(seed=100)() for _ in range(1)] + good_rolls[1:]

Bad rolls: [2, 2, 2, 2, 2, 2, 2, 2, 2, 2]
Good reproducible rolls: [2, 4, 4, 2, 6, 4, 6, 3, 4, 5]


### Explanation

Seeding inside the random function resets the generator every time, so it repeats the first result forever. Usually, seed once at the start of an experiment, not inside every draw.

## Problem 7: Reproducible Monte Carlo Estimate

Use randomness to estimate the value of π.

A point `(x, y)` is sampled uniformly from the square `[-1, 1] × [-1, 1]`. The probability that it lies inside the unit circle is approximately `π / 4`.

**Task**

1. Write a reproducible Monte Carlo estimator for π.
2. Show that the same seed returns the same estimate.
3. Show that increasing `n` usually improves the estimate.

### Solution

In [10]:
def estimate_pi(n, seed=None):
    rng = random.Random(seed)
    inside = 0
    
    for _ in range(n):
        x = rng.uniform(-1, 1)
        y = rng.uniform(-1, 1)
        
        if x * x + y * y <= 1:
            inside += 1
            
    return 4 * inside / n


small = estimate_pi(1_000, seed=123)
large = estimate_pi(100_000, seed=123)

assert estimate_pi(10_000, seed=999) == estimate_pi(10_000, seed=999)

print("Estimate with 1,000 samples:", small)
print("Estimate with 100,000 samples:", large)
print("Absolute error, small:", abs(small - 3.141592653589793))
print("Absolute error, large:", abs(large - 3.141592653589793))

Estimate with 1,000 samples: 3.156
Estimate with 100,000 samples: 3.13688
Absolute error, small: 0.014407346410207023
Absolute error, large: 0.004712653589793003


### Explanation

Monte Carlo methods are random simulations. Seeding makes them reproducible, which is essential for debugging, grading, and scientific reporting.

## Problem 8: Reproducible Gaussian Simulation

Simulate 10,000 measurements from a normal distribution with mean `50` and standard deviation `8`.

**Task**

1. Use `random.gauss`.
2. Make the result reproducible.
3. Compute the empirical mean and population standard deviation.
4. Check that they are close to the requested parameters.

### Solution

In [11]:
def gaussian_measurements(n, mu, sigma, seed=None):
    rng = random.Random(seed)
    return [rng.gauss(mu, sigma) for _ in range(n)]


measurements = gaussian_measurements(10_000, mu=50, sigma=8, seed=2024)

empirical_mean = mean(measurements)
empirical_std = pstdev(measurements)

print("Empirical mean:", empirical_mean)
print("Empirical population standard deviation:", empirical_std)

assert abs(empirical_mean - 50) < 0.3
assert abs(empirical_std - 8) < 0.3
assert measurements == gaussian_measurements(10_000, mu=50, sigma=8, seed=2024)

Empirical mean: 50.08757732924873
Empirical population standard deviation: 8.108428735488296


### Explanation

The sample statistics will not be exactly equal to the target parameters, but with a large sample they should be close.

## Problem 9: Randomized Test Case Generator

Create a reproducible test-case generator for sorting algorithms.

**Task**

Write `generate_sort_cases(seed)` that returns a dictionary with:

- `"empty"`: an empty list
- `"single"`: a one-item list
- `"small_random"`: 10 random integers from -20 to 20
- `"many_duplicates"`: 30 random integers from 0 to 5
- `"already_sorted"`: sorted version of 15 random integers
- `"reverse_sorted"`: reverse-sorted version of 15 random integers

Then verify that all cases sort correctly using Python's built-in `sorted`.

### Solution

In [12]:
def generate_sort_cases(seed=None):
    rng = random.Random(seed)
    
    base_for_sorted = [rng.randint(-100, 100) for _ in range(15)]
    base_for_reverse = [rng.randint(-100, 100) for _ in range(15)]
    
    return {
        "empty": [],
        "single": [rng.randint(-20, 20)],
        "small_random": [rng.randint(-20, 20) for _ in range(10)],
        "many_duplicates": [rng.randint(0, 5) for _ in range(30)],
        "already_sorted": sorted(base_for_sorted),
        "reverse_sorted": sorted(base_for_reverse, reverse=True),
    }


cases1 = generate_sort_cases(seed=77)
cases2 = generate_sort_cases(seed=77)

assert cases1 == cases2

for name, case in cases1.items():
    assert sorted(case) == sorted(list(case))
    print(name, "->", case)

empty -> []
single -> [-4]
small_random -> [14, 10, -7, -6, -8, -16, -15, -8, 15, 8]
many_duplicates -> [4, 5, 4, 0, 0, 5, 5, 1, 5, 2, 0, 0, 4, 3, 0, 4, 0, 2, 3, 5, 4, 0, 2, 4, 0, 3, 4, 4, 4, 1]
already_sorted -> [-71, -63, -51, -50, -40, -39, -36, -26, -17, 21, 42, 42, 56, 61, 69]
reverse_sorted -> [41, 29, 28, 27, -1, -16, -18, -29, -52, -54, -56, -70, -78, -93, -100]


### Explanation

Randomized test data is much more useful when failures are reproducible. The seed lets you recreate the exact failing input.

## Problem 10: Independent Streams for Simulation Components

You are simulating customers at a store.

Each customer has:

- an arrival gap sampled from `random.expovariate(1 / 5)`
- a basket size sampled from `random.randint(1, 20)`

**Task**

Write a simulation where arrival gaps and basket sizes use separate random streams. Then show that changing the basket-size seed does not affect arrival gaps.

### Solution

In [13]:
def simulate_customers(n, arrival_seed=None, basket_seed=None):
    arrival_rng = random.Random(arrival_seed)
    basket_rng = random.Random(basket_seed)
    
    customers = []
    
    for customer_id in range(1, n + 1):
        arrival_gap = arrival_rng.expovariate(1 / 5)
        basket_size = basket_rng.randint(1, 20)
        
        customers.append({
            "customer_id": customer_id,
            "arrival_gap": arrival_gap,
            "basket_size": basket_size,
        })
        
    return customers


sim_a = simulate_customers(5, arrival_seed=10, basket_seed=100)
sim_b = simulate_customers(5, arrival_seed=10, basket_seed=999)

arrival_gaps_a = [row["arrival_gap"] for row in sim_a]
arrival_gaps_b = [row["arrival_gap"] for row in sim_b]

basket_sizes_a = [row["basket_size"] for row in sim_a]
basket_sizes_b = [row["basket_size"] for row in sim_b]

assert arrival_gaps_a == arrival_gaps_b
assert basket_sizes_a != basket_sizes_b

print("Simulation A:", sim_a)
print("Simulation B:", sim_b)

Simulation A: [{'customer_id': 1, 'arrival_gap': 4.236186249169292, 'basket_size': 5}, {'customer_id': 2, 'arrival_gap': 2.8008589407817674, 'basket_size': 15}, {'customer_id': 3, 'arrival_gap': 4.314831708765586, 'basket_size': 15}, {'customer_id': 4, 'arrival_gap': 1.1539777172426882, 'basket_size': 6}, {'customer_id': 5, 'arrival_gap': 8.39183030572058, 'basket_size': 13}]
Simulation B: [{'customer_id': 1, 'arrival_gap': 4.236186249169292, 'basket_size': 3}, {'customer_id': 2, 'arrival_gap': 2.8008589407817674, 'basket_size': 19}, {'customer_id': 3, 'arrival_gap': 4.314831708765586, 'basket_size': 19}, {'customer_id': 4, 'arrival_gap': 1.1539777172426882, 'basket_size': 18}, {'customer_id': 5, 'arrival_gap': 8.39183030572058, 'basket_size': 16}]


### Explanation

Separate streams make complex simulations easier to debug. One component can change without accidentally changing every other random sequence.

## Problem 11: Save and Restore Random State

The `random` module allows you to save and restore generator state using `getstate()` and `setstate()`.

**Task**

1. Generate five values.
2. Save the random state.
3. Generate five more values.
4. Restore the saved state.
5. Generate five values again.
6. Prove that the regenerated values match the previous five.

### Solution

In [14]:
rng = random.Random(123)

first_five = [rng.random() for _ in range(5)]

saved_state = rng.getstate()

next_five = [rng.random() for _ in range(5)]

rng.setstate(saved_state)

regenerated_next_five = [rng.random() for _ in range(5)]

print("First five:", first_five)
print("Next five:", next_five)
print("Regenerated next five:", regenerated_next_five)

assert next_five == regenerated_next_five

First five: [0.052363598850944326, 0.08718667752263232, 0.4072417636703983, 0.10770023493843905, 0.9011988779516946]
Next five: [0.0381536661023224, 0.5362020400339269, 0.33219769850967984, 0.8520866189293687, 0.1596623967219699]
Regenerated next five: [0.0381536661023224, 0.5362020400339269, 0.33219769850967984, 0.8520866189293687, 0.1596623967219699]


### Explanation

Saving state is useful when you need to pause, branch, replay, or debug a random process from a specific point.

## Problem 12: Build a Reproducible Experiment Object

Design a small class called `RandomExperiment`.

Requirements:

1. It stores its own `random.Random` instance.
2. It has a method `roll_die()`.
3. It has a method `sample_names(names, k)` that samples without replacement.
4. It has a method `reset(seed)` that resets the experiment.
5. Two experiments with the same seed should produce the same outputs.

### Solution

In [15]:
class RandomExperiment:
    def __init__(self, seed=None):
        self.seed = seed
        self.rng = random.Random(seed)
        
    def roll_die(self):
        return self.rng.randint(1, 6)
    
    def sample_names(self, names, k):
        if k > len(names):
            raise ValueError("k cannot be larger than the number of names")
        return self.rng.sample(names, k)
    
    def reset(self, seed=None):
        self.seed = seed
        self.rng.seed(seed)


names = ["Ada", "Grace", "Linus", "Guido", "Barbara"]

exp1 = RandomExperiment(seed=555)
exp2 = RandomExperiment(seed=555)

outputs1 = [
    exp1.roll_die(),
    exp1.roll_die(),
    exp1.sample_names(names, 3),
    exp1.roll_die(),
]

outputs2 = [
    exp2.roll_die(),
    exp2.roll_die(),
    exp2.sample_names(names, 3),
    exp2.roll_die(),
]

print(outputs1)
print(outputs2)

assert outputs1 == outputs2

exp1.reset(seed=555)
assert exp1.roll_die() == RandomExperiment(seed=555).roll_die()

[2, 3, ['Grace', 'Barbara', 'Linus'], 6]
[2, 3, ['Grace', 'Barbara', 'Linus'], 6]


### Explanation

Encapsulating randomness inside an object avoids global-state bugs and makes experiments easier to reproduce and test.

## Best-practice checklist

When working with pseudo-randomness:

1. Seed once per experiment, not once per random draw.
2. Prefer `random.Random(seed)` for independent reproducible streams.
3. Avoid hidden dependence on the global `random` state in reusable functions.
4. Copy lists before shuffling if the caller's original data should remain unchanged.
5. Store the seed alongside experiment results.
6. For tests, use fixed seeds and assert properties, not just exact values.
7. Remember that reproducible does not mean statistically perfect.
8. Use `Counter` for frequency analysis instead of repeatedly calling `list.count`.